# JRC European flood hazard (EFHM) — catalog explorer (no network)

The `jrc-flood` backend serves the JRC European Flood Hazard Map: river-flood
**water depth (m)** for a set of **return periods** over Europe and the
Mediterranean. This notebook inspects the catalog and the URL / pixel-window
helpers without touching the network.

## The catalog

One product, addressed by return period (not by a `variables` axis).

In [ ]:
from earthlens.jrc_flood import Catalog

catalog = Catalog()
print("datasets:", list(catalog.datasets))
row = catalog.get("efhm")
print("band:", row.band, "| units:", row.units, "| CRS:", row.crs)
print("return periods:", row.return_periods)
print("licence:", catalog.license_id)

## How a small AOI stays cheap

Each return period is one whole-Europe GeoTIFF (~23 GB uncompressed). The backend
never reads it whole: it opens the file lazily and reads only the AOI's pixel
window over `/vsicurl` (HTTP range requests) via pyramids' `Dataset.crop(bbox=)`.
Building the request — the per-return-period URL below — is offline; only
`.download()` touches the network.

In [ ]:
from earthlens.jrc_flood._helpers import efhm_url

# One whole-Europe ~23 GB GeoTIFF per return period; a small AOI reads only its
# pixel window via pyramids' Dataset.crop(bbox=) over /vsicurl (a few hundred KB).
for rp in (100, 200, 500):
    print(f"RP{rp}:", efhm_url(rp))

## Next

See **EFHM quickstart** for a real windowed download and map.